# Complemento NeuroKit2 — WESAD | checkpoints incrementales

**Objetivo.** Sin recalcular las 869 columnas previas, extraer exclusivamente las nuevas características basadas en NeuroKit2 para **ECG y EDA** y anexarlas a una copia de cada checkpoint de sujeto.

- **Originales, sin modificar:** `wesad_feature_output/checkpoints/S*_features.pkl`
- **Nuevos (solo NeuroKit2):** `wesad_feature_output/checkpoints_neurokit/S*_neurokit_features.pkl`
- **Fusionados:** `wesad_feature_output/checkpoints_completos/S*_features_full.pkl`
- **Consolidado:** `wesad_feature_output/wesad_processed_full_neurokit.pkl`

**Requisitos:** disponer de la carpeta original `WESAD/S2/S2.pkl`, etc., junto a `wesad_feature_output`. Los datos crudos son indispensables: no se pueden reconstruir las características cardíacas o electrodérmicas a partir de la matriz de *features*. **Nunca pongas `HAS_NEUROKIT=True` manualmente**; esta libreta comprueba el import real.

**Importante:** los indicadores de HRV espectral derivados de una ventana de 60 segundos deben interpretarse con cautela, particularmente VLF. Si no hay suficientes picos o falla una función, se reporta mediante el control de cobertura. El proceso puede tardar mucho con ventanas desplazadas 250 ms.

## 1. Instalar dependencias obligatorias (ejecutar esta celda primero)

In [1]:
%pip install --upgrade "neurokit2>=0.2.10" "antropy>=0.1.6" "nolds>=0.6.0" "PyWavelets>=1.4" "scipy>=1.10" "numpy>=1.23" "pandas>=1.5" "tqdm>=4.60"

  Using cached numpy-2.5.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.6-cp313-cp313-win_amd64.whl.metadata (19 kB)
Note: you may need to restart the kernel to use updated packages.


## 2. Importación y verificación estricta

In [2]:
import os, gc, pickle, json, warnings, math
from pathlib import Path
from typing import Dict, Sequence, Tuple, List, Optional
import numpy as np
import pandas as pd
from scipy import signal, stats
from scipy.integrate import trapezoid
from scipy.spatial.distance import pdist, squareform
from scipy.stats import entropy as scipy_entropy
from tqdm.auto import tqdm
import neurokit2 as nk
import pywt
import antropy as ant
import nolds
HAS_ANTROPY=True
HAS_NOLDS=True
HAS_NEUROKIT=True
HAS_PYWT=True
print('NeuroKit2:',nk.__version__,'| PyWavelets:',pywt.__version__)
assert HAS_NEUROKIT and HAS_PYWT, 'Instalación incompleta. Reinicia el kernel y repite la importación.'
# Se usan las mismas funciones auxiliares del notebook original. AntroPy/nolds siguen siendo opcionales.

NeuroKit2: 0.2.13 | PyWavelets: 1.8.0


In [3]:
print({
    "NeuroKit2": HAS_NEUROKIT,
    "PyWavelets": HAS_PYWT,
    "AntroPy": HAS_ANTROPY,
    "nolds": HAS_NOLDS
})

{'NeuroKit2': True, 'PyWavelets': True, 'AntroPy': True, 'nolds': True}


## 3. Configuración de rutas, sujetos y opciones

In [4]:
PROJECT_ROOT = Path.cwd()  # Debe contener WESAD/ y wesad_feature_output/
BASE_PATH = PROJECT_ROOT / 'WESAD'
OUTPUT_DIR = PROJECT_ROOT / 'wesad_feature_output'
ORIGINAL_DIR = OUTPUT_DIR / 'checkpoints'
EXTRA_DIR = OUTPUT_DIR / 'checkpoints_neurokit'
MERGED_DIR = OUTPUT_DIR / 'checkpoints_completos'
FINAL_OUTPUT = OUTPUT_DIR / 'wesad_processed_full_neurokit.pkl'
SUBJECTS = ['S2','S3','S4','S5','S6','S7','S8','S9','S10','S11','S13','S14','S15','S16','S17']
SAMPLING_RATE = 700.0
ENABLE_WAVELETS=True
ENABLE_FRACTALS=True
ENABLE_ENTROPIES=True
ENABLE_RQA=True
NONLINEAR_MAX_POINTS=1500
RQA_MAX_POINTS=400
OUTPUT_DTYPE=np.float32
# Si alguna ventana falla, conservamos columnas con NaN para no perder correspondencia.
# Para rehacer un checkpoint extra: cambia a True; NO afecta los checkpoints originales.
OVERWRITE_EXTRA=False
# Para una prueba rápida por sujeto, escribe ['S2'] y pon TEST_MAX_WINDOWS=5.
SUBJECTS_TO_PROCESS=SUBJECTS
TEST_MAX_WINDOWS=None   # None = todas las ventanas; si es entero, no fusionar como definitivo.
EXTRA_DIR.mkdir(parents=True,exist_ok=True)
MERGED_DIR.mkdir(parents=True,exist_ok=True)
assert ORIGINAL_DIR.exists(),f'No existe {ORIGINAL_DIR}'
assert BASE_PATH.exists(),f'No existe {BASE_PATH}'
print('Original:',ORIGINAL_DIR, 'Nuevos:',EXTRA_DIR, 'Fusionados:',MERGED_DIR)

Original: C:\Users\Victor\Documents\THESIS_WESAD\wesad_feature_output\checkpoints Nuevos: C:\Users\Victor\Documents\THESIS_WESAD\wesad_feature_output\checkpoints_neurokit Fusionados: C:\Users\Victor\Documents\THESIS_WESAD\wesad_feature_output\checkpoints_completos


In [14]:
import warnings
warnings.filterwarnings("ignore")

## 4. Funciones auxiliares idénticas a la libreta original

In [15]:
def load_wesad_subject(base_path: Path, subject: str) -> Dict:
    file_path = base_path / subject / f"{subject}.pkl"

    if not file_path.exists():
        raise FileNotFoundError(f"No se encontró: {file_path}")

    with open(file_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    if "signal" not in data or "chest" not in data["signal"]:
        raise KeyError(f"{subject} no contiene signal/chest")

    if "label" not in data:
        raise KeyError(f"{subject} no contiene label")

    return data


def inspect_subject_structure(base_path: Path, subject: str = "S2") -> None:
    data = load_wesad_subject(base_path, subject)

    print("Claves principales:", data.keys())
    print("Claves signal:", data["signal"].keys())
    print("Claves chest:", data["signal"]["chest"].keys())
    print("Labels:", np.unique(data["label"], return_counts=True))

    for name, values in data["signal"]["chest"].items():
        print(name, np.asarray(values).shape, np.asarray(values).dtype)

    del data
    gc.collect()


# Ejecuta esta celda para verificar la estructura.
inspect_subject_structure(BASE_PATH, "S2")

Claves principales: dict_keys(['signal', 'label', 'subject'])
Claves signal: dict_keys(['chest', 'wrist'])
Claves chest: dict_keys(['ACC', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp'])
Labels: (array([0, 1, 2, 3, 4, 6, 7], dtype=int32), array([2142701,  800800,  430500,  253400,  537599,   45500,   44800]))
ACC (4255300, 3) float64
ECG (4255300, 1) float64
EMG (4255300, 1) float64
EDA (4255300, 1) float64
Temp (4255300, 1) float32
Resp (4255300, 1) float64


In [16]:
EPSILON = 1e-12


def sanitize_signal(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64).reshape(-1)

    if x.size == 0:
        return x

    finite = np.isfinite(x)

    if finite.all():
        return x

    if not finite.any():
        return np.zeros_like(x)

    idx = np.arange(x.size)
    x[~finite] = np.interp(idx[~finite], idx[finite], x[finite])
    return x


def downsample_for_nonlinear(x: np.ndarray, max_points: int = NONLINEAR_MAX_POINTS) -> np.ndarray:
    x = sanitize_signal(x)

    if x.size <= max_points:
        return x

    idx = np.linspace(0, x.size - 1, max_points).astype(np.int64)
    return x[idx]


def safe_float(value, default=np.nan) -> float:
    try:
        value = float(value)
        return value if np.isfinite(value) else default
    except Exception:
        return default


def safe_skew(x: np.ndarray) -> float:
    x = sanitize_signal(x)
    if x.size < 3 or np.std(x) < EPSILON:
        return 0.0
    return safe_float(stats.skew(x, bias=False), 0.0)


def safe_kurtosis(x: np.ndarray) -> float:
    x = sanitize_signal(x)
    if x.size < 4 or np.std(x) < EPSILON:
        return 0.0
    return safe_float(stats.kurtosis(x, fisher=True, bias=False), 0.0)


def linear_slope(x: np.ndarray, sampling_rate: float) -> float:
    x = sanitize_signal(x)
    if x.size < 2:
        return 0.0
    t = np.arange(x.size, dtype=np.float64) / sampling_rate
    return safe_float(np.polyfit(t, x, 1)[0], 0.0)


def zero_crossing_rate(x: np.ndarray) -> float:
    x = sanitize_signal(x)
    if x.size < 2:
        return 0.0
    centered = x - np.mean(x)
    return float(np.mean(centered[:-1] * centered[1:] < 0))


def mean_crossing_rate(x: np.ndarray) -> float:
    return zero_crossing_rate(x)


def waveform_length(x: np.ndarray) -> float:
    x = sanitize_signal(x)
    return float(np.sum(np.abs(np.diff(x)))) if x.size > 1 else 0.0


def slope_sign_changes(x: np.ndarray, threshold: float = 0.0) -> float:
    x = sanitize_signal(x)
    if x.size < 3:
        return 0.0
    d1 = np.diff(x)
    changes = ((d1[:-1] * d1[1:]) < 0) & (
        (np.abs(d1[:-1]) >= threshold) | (np.abs(d1[1:]) >= threshold)
    )
    return float(np.sum(changes))


def willison_amplitude(x: np.ndarray, threshold: Optional[float] = None) -> float:
    x = sanitize_signal(x)
    if x.size < 2:
        return 0.0
    if threshold is None:
        threshold = 0.1 * np.std(x)
    return float(np.sum(np.abs(np.diff(x)) > threshold))


def teager_kaiser_energy(x: np.ndarray) -> Tuple[float, float]:
    x = sanitize_signal(x)
    if x.size < 3:
        return 0.0, 0.0
    tkeo = x[1:-1] ** 2 - x[:-2] * x[2:]
    return safe_float(np.mean(tkeo), 0.0), safe_float(np.std(tkeo), 0.0)


def spectral_features(
    x: np.ndarray,
    sampling_rate: float,
    prefix: str = "",
    bands: Optional[Dict[str, Tuple[float, float]]] = None
) -> Dict[str, float]:
    x = sanitize_signal(x)

    base = {
        f"{prefix}dominant_frequency": 0.0,
        f"{prefix}mean_frequency": 0.0,
        f"{prefix}median_frequency": 0.0,
        f"{prefix}spectral_entropy": 0.0,
        f"{prefix}total_spectral_power": 0.0,
        f"{prefix}spectral_centroid": 0.0,
        f"{prefix}spectral_spread": 0.0,
        f"{prefix}spectral_skewness": 0.0,
        f"{prefix}spectral_kurtosis": 0.0,
    }

    if x.size < 8 or np.std(x) < EPSILON:
        if bands:
            for band_name in bands:
                base[f"{prefix}power_{band_name}"] = 0.0
                base[f"{prefix}relative_power_{band_name}"] = 0.0
        return base

    f, psd = signal.welch(
        x,
        fs=sampling_rate,
        nperseg=min(2048, x.size)
    )

    psd = np.maximum(psd, 0.0)
    psd_sum = np.sum(psd)
    total_power = trapezoid(psd, f) if f.size > 1 else 0.0

    if psd_sum <= EPSILON:
        return base

    p = psd / psd_sum
    dom = f[int(np.argmax(psd))]
    mean_f = np.sum(f * psd) / psd_sum
    cum = np.cumsum(psd)
    median_f = f[min(np.searchsorted(cum, cum[-1] / 2), f.size - 1)]
    spec_entropy = -np.sum(p * np.log2(p + EPSILON)) / np.log2(max(len(p), 2))
    spread = np.sqrt(np.sum(((f - mean_f) ** 2) * p))

    if spread > EPSILON:
        skew = np.sum(((f - mean_f) / spread) ** 3 * p)
        kurt = np.sum(((f - mean_f) / spread) ** 4 * p)
    else:
        skew = 0.0
        kurt = 0.0

    out = {
        f"{prefix}dominant_frequency": safe_float(dom, 0.0),
        f"{prefix}mean_frequency": safe_float(mean_f, 0.0),
        f"{prefix}median_frequency": safe_float(median_f, 0.0),
        f"{prefix}spectral_entropy": safe_float(spec_entropy, 0.0),
        f"{prefix}total_spectral_power": safe_float(total_power, 0.0),
        f"{prefix}spectral_centroid": safe_float(mean_f, 0.0),
        f"{prefix}spectral_spread": safe_float(spread, 0.0),
        f"{prefix}spectral_skewness": safe_float(skew, 0.0),
        f"{prefix}spectral_kurtosis": safe_float(kurt, 0.0),
    }

    if bands:
        for band_name, (low, high) in bands.items():
            mask = (f >= low) & (f < high)
            power = trapezoid(psd[mask], f[mask]) if np.sum(mask) > 1 else 0.0
            out[f"{prefix}power_{band_name}"] = safe_float(power, 0.0)
            out[f"{prefix}relative_power_{band_name}"] = safe_float(
                power / (total_power + EPSILON), 0.0
            )

    return out


def general_features(
    x: np.ndarray,
    sampling_rate: float,
    prefix: str = ""
) -> Dict[str, float]:
    x = sanitize_signal(x)

    if x.size == 0:
        return {}

    q01, q05, q10, q25, q50, q75, q90, q95, q99 = np.percentile(
        x, [1, 5, 10, 25, 50, 75, 90, 95, 99]
    )

    mean = np.mean(x)
    std = np.std(x)
    abs_x = np.abs(x)

    out = {
        f"{prefix}mean": mean,
        f"{prefix}median": q50,
        f"{prefix}std": std,
        f"{prefix}variance": np.var(x),
        f"{prefix}min": np.min(x),
        f"{prefix}max": np.max(x),
        f"{prefix}range": np.ptp(x),
        f"{prefix}q01": q01,
        f"{prefix}q05": q05,
        f"{prefix}q10": q10,
        f"{prefix}q25": q25,
        f"{prefix}q75": q75,
        f"{prefix}q90": q90,
        f"{prefix}q95": q95,
        f"{prefix}q99": q99,
        f"{prefix}iqr": q75 - q25,
        f"{prefix}mad_mean": np.mean(np.abs(x - mean)),
        f"{prefix}mad_median": stats.median_abs_deviation(x),
        f"{prefix}rms": np.sqrt(np.mean(x ** 2)),
        f"{prefix}mean_absolute_value": np.mean(abs_x),
        f"{prefix}energy": np.sum(x ** 2),
        f"{prefix}mean_power": np.mean(x ** 2),
        f"{prefix}auc_absolute": trapezoid(abs_x, dx=1 / sampling_rate),
        f"{prefix}auc_signed": trapezoid(x, dx=1 / sampling_rate),
        f"{prefix}skewness": safe_skew(x),
        f"{prefix}kurtosis": safe_kurtosis(x),
        f"{prefix}slope": linear_slope(x, sampling_rate),
        f"{prefix}delta": x[-1] - x[0],
        f"{prefix}coefficient_variation": std / (abs(mean) + EPSILON),
        f"{prefix}zero_crossing_rate": zero_crossing_rate(x),
        f"{prefix}mean_crossing_rate": mean_crossing_rate(x),
        f"{prefix}waveform_length": waveform_length(x),
        f"{prefix}crest_factor": np.max(abs_x) / (np.sqrt(np.mean(x ** 2)) + EPSILON),
        f"{prefix}shape_factor": np.sqrt(np.mean(x ** 2)) / (np.mean(abs_x) + EPSILON),
        f"{prefix}impulse_factor": np.max(abs_x) / (np.mean(abs_x) + EPSILON),
        f"{prefix}clearance_factor": np.max(abs_x) / (np.mean(np.sqrt(abs_x + EPSILON)) ** 2 + EPSILON),
    }

    return {k: safe_float(v, 0.0) for k, v in out.items()}

In [17]:
def approximate_entropy(x: np.ndarray, m: int = 2, r_ratio: float = 0.2) -> float:
    x = downsample_for_nonlinear(x)
    n = len(x)

    if n < m + 2 or np.std(x) < EPSILON:
        return 0.0

    r = r_ratio * np.std(x)

    def _phi(mm):
        patterns = np.array([x[i:i + mm] for i in range(n - mm + 1)])
        distances = np.max(np.abs(patterns[:, None, :] - patterns[None, :, :]), axis=2)
        counts = np.mean(distances <= r, axis=0)
        return np.mean(np.log(counts + EPSILON))

    return safe_float(_phi(m) - _phi(m + 1), np.nan)


def sample_entropy_feature(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)

    if x.size < 20 or np.std(x) < EPSILON:
        return 0.0

    try:
        if HAS_ANTROPY:
            return safe_float(ant.sample_entropy(x), np.nan)
        if HAS_NEUROKIT:
            value, _ = nk.entropy_sample(x)
            return safe_float(value, np.nan)
    except Exception:
        pass

    return np.nan


def permutation_entropy_feature(x: np.ndarray, order: int = 3, delay: int = 1) -> float:
    x = downsample_for_nonlinear(x)

    if x.size < order * delay + 2:
        return 0.0

    try:
        if HAS_ANTROPY:
            return safe_float(
                ant.perm_entropy(x, order=order, delay=delay, normalize=True),
                np.nan
            )
    except Exception:
        pass

    patterns = {}
    for i in range(x.size - delay * (order - 1)):
        window = x[i:i + delay * order:delay]
        key = tuple(np.argsort(window))
        patterns[key] = patterns.get(key, 0) + 1

    counts = np.asarray(list(patterns.values()), dtype=np.float64)
    probs = counts / counts.sum()
    pe = -np.sum(probs * np.log2(probs + EPSILON))
    return safe_float(pe / np.log2(math.factorial(order)), np.nan)


def lempel_ziv_complexity_feature(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)
    if x.size < 4:
        return 0.0

    binary = (x > np.median(x)).astype(np.uint8)
    s = "".join(binary.astype(str))

    i, k, l, c, k_max = 0, 1, 1, 1, 1
    n = len(s)

    while True:
        if i + k >= n or l + k >= n:
            c += 1
            break
        if s[i + k] == s[l + k]:
            k += 1
            if l + k >= n:
                c += 1
                break
        else:
            if k > k_max:
                k_max = k
            i += 1
            if i == l:
                c += 1
                l += k_max
                if l + 1 >= n:
                    break
                i, k, k_max = 0, 1, 1
            else:
                k = 1

    normalization = n / max(np.log2(n), EPSILON)
    return safe_float(c / normalization, np.nan)


def petrosian_fd(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)
    if x.size < 3:
        return 0.0
    diff = np.diff(x)
    n_delta = np.sum(diff[1:] * diff[:-1] < 0)
    n = x.size
    denominator = np.log10(n) + np.log10(n / (n + 0.4 * n_delta + EPSILON))
    return safe_float(np.log10(n) / (denominator + EPSILON), np.nan)


def katz_fd(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)
    if x.size < 3:
        return 0.0
    distances = np.abs(np.diff(x))
    length = np.sum(distances)
    diameter = np.max(np.abs(x - x[0]))
    n = x.size
    if length <= EPSILON or diameter <= EPSILON:
        return 1.0
    return safe_float(
        np.log10(n) / (np.log10(n) + np.log10(diameter / length + EPSILON)),
        np.nan
    )


def higuchi_fd(x: np.ndarray, kmax: int = 8) -> float:
    x = downsample_for_nonlinear(x)

    if x.size < 20:
        return np.nan

    if HAS_ANTROPY:
        try:
            return safe_float(ant.higuchi_fd(x, kmax=kmax), np.nan)
        except Exception:
            pass

    n = len(x)
    lengths = []

    for k in range(1, kmax + 1):
        lk = []
        for m in range(k):
            idx = np.arange(1, int(np.floor((n - m - 1) / k)) + 1)
            if idx.size == 0:
                continue
            distance = np.sum(np.abs(x[m + idx * k] - x[m + (idx - 1) * k]))
            norm = (n - 1) / (idx.size * k)
            lk.append(distance * norm / k)
        lengths.append(np.mean(lk) if lk else np.nan)

    lengths = np.asarray(lengths)
    valid = np.isfinite(lengths) & (lengths > 0)
    if np.sum(valid) < 2:
        return np.nan

    return safe_float(
        np.polyfit(np.log(1.0 / np.arange(1, kmax + 1)[valid]), np.log(lengths[valid]), 1)[0],
        np.nan
    )


def dfa_feature(x: np.ndarray) -> float:
    x = downsample_for_nonlinear(x)

    if x.size < 32 or np.std(x) < EPSILON:
        return np.nan

    try:
        if HAS_NOLDS:
            return safe_float(nolds.dfa(x), np.nan)
        if HAS_ANTROPY:
            return safe_float(ant.detrended_fluctuation(x), np.nan)
    except Exception:
        pass

    return np.nan


def rqa_features(x: np.ndarray, max_points: int = RQA_MAX_POINTS) -> Dict[str, float]:
    x = downsample_for_nonlinear(x, max_points=max_points)

    defaults = {
        "rqa_recurrence_rate": np.nan,
        "rqa_determinism": np.nan,
        "rqa_laminarity": np.nan,
        "rqa_diagonal_entropy": np.nan,
    }

    if x.size < 30 or np.std(x) < EPSILON:
        return defaults

    try:
        z = (x - np.mean(x)) / (np.std(x) + EPSILON)
        distances = squareform(pdist(z[:, None], metric="euclidean"))
        epsilon = np.quantile(distances, 0.10)
        recurrence = distances <= epsilon
        np.fill_diagonal(recurrence, False)

        rr = np.mean(recurrence)

        diag_lengths = []
        for offset in range(-len(x) + 1, len(x)):
            diag = np.diag(recurrence, k=offset).astype(np.int8)
            if diag.size < 2:
                continue
            padded = np.r_[0, diag, 0]
            starts = np.where(np.diff(padded) == 1)[0]
            ends = np.where(np.diff(padded) == -1)[0]
            diag_lengths.extend((ends - starts).tolist())

        vert_lengths = []
        for col in range(recurrence.shape[1]):
            colv = recurrence[:, col].astype(np.int8)
            padded = np.r_[0, colv, 0]
            starts = np.where(np.diff(padded) == 1)[0]
            ends = np.where(np.diff(padded) == -1)[0]
            vert_lengths.extend((ends - starts).tolist())

        diag_lengths = np.asarray(diag_lengths)
        vert_lengths = np.asarray(vert_lengths)

        det = (
            np.sum(diag_lengths[diag_lengths >= 2]) / (np.sum(diag_lengths) + EPSILON)
            if diag_lengths.size else 0.0
        )

        lam = (
            np.sum(vert_lengths[vert_lengths >= 2]) / (np.sum(vert_lengths) + EPSILON)
            if vert_lengths.size else 0.0
        )

        valid_diag = diag_lengths[diag_lengths >= 2]
        if valid_diag.size:
            counts = np.bincount(valid_diag)
            probs = counts[counts > 0] / np.sum(counts)
            diag_entropy = -np.sum(probs * np.log(probs + EPSILON))
        else:
            diag_entropy = 0.0

        return {
            "rqa_recurrence_rate": safe_float(rr),
            "rqa_determinism": safe_float(det),
            "rqa_laminarity": safe_float(lam),
            "rqa_diagonal_entropy": safe_float(diag_entropy),
        }

    except Exception:
        return defaults


def nonlinear_features(x: np.ndarray, prefix: str = "") -> Dict[str, float]:
    out = {}

    if ENABLE_ENTROPIES:
        out.update({
            f"{prefix}sample_entropy": sample_entropy_feature(x),
            f"{prefix}approximate_entropy": approximate_entropy(x),
            f"{prefix}permutation_entropy": permutation_entropy_feature(x),
            f"{prefix}lempel_ziv_complexity": lempel_ziv_complexity_feature(x),
        })

    if ENABLE_FRACTALS:
        out.update({
            f"{prefix}petrosian_fd": petrosian_fd(x),
            f"{prefix}katz_fd": katz_fd(x),
            f"{prefix}higuchi_fd": higuchi_fd(x),
            f"{prefix}dfa": dfa_feature(x),
        })

    if ENABLE_RQA:
        out.update({
            f"{prefix}{key}": value
            for key, value in rqa_features(x).items()
        })

    return out

In [18]:
def wavelet_features(
    x: np.ndarray,
    prefix: str = "",
    wavelet: str = "db4",
    level: int = 4
) -> Dict[str, float]:
    if not ENABLE_WAVELETS or not HAS_PYWT:
        return {}

    x = sanitize_signal(x)

    if x.size < 32:
        return {}

    try:
        max_level = pywt.dwt_max_level(x.size, pywt.Wavelet(wavelet).dec_len)
        use_level = max(1, min(level, max_level))
        coeffs = pywt.wavedec(x, wavelet=wavelet, level=use_level)

        energies = np.asarray([np.sum(c ** 2) for c in coeffs], dtype=np.float64)
        total = np.sum(energies) + EPSILON

        out = {}
        for i, c in enumerate(coeffs):
            label = "A" if i == 0 else f"D{use_level - i + 1}"
            energy = energies[i]
            p = (c ** 2) / (np.sum(c ** 2) + EPSILON)
            coeff_entropy = -np.sum(p * np.log2(p + EPSILON))

            out[f"{prefix}wavelet_{label}_energy"] = safe_float(energy, 0.0)
            out[f"{prefix}wavelet_{label}_relative_energy"] = safe_float(energy / total, 0.0)
            out[f"{prefix}wavelet_{label}_mean_abs"] = safe_float(np.mean(np.abs(c)), 0.0)
            out[f"{prefix}wavelet_{label}_std"] = safe_float(np.std(c), 0.0)
            out[f"{prefix}wavelet_{label}_entropy"] = safe_float(coeff_entropy, 0.0)

        return out
    except Exception:
        return {}

## 5. Extracción adicional ECG y EDA (sin volver a calcular las columnas crudas)

Estos extractores conservan la **rama NeuroKit2 del notebook original**: detección de picos R, HRV y separación tónica/fásica. Solo se omite el bloque inicial de características crudas, porque ya existe en los checkpoints.

In [19]:
def extract_ecg_neurokit_only(ecg: np.ndarray, sampling_rate: float = 700.0) -> Dict[str, float]:
    ecg = sanitize_signal(ecg)
    out = {}
    try:
        cleaned = nk.ecg_clean(ecg, sampling_rate=sampling_rate)
        _, info = nk.ecg_peaks(cleaned, sampling_rate=sampling_rate)
        rpeaks = np.asarray(info.get("ECG_R_Peaks", []), dtype=np.int64)

        out["ecg_rpeak_count"] = float(rpeaks.size)
        out["ecg_rpeak_rate_per_minute"] = float(
            rpeaks.size / (ecg.size / sampling_rate / 60.0 + EPSILON)
        )

        if rpeaks.size >= 3:
            rr_ms = np.diff(rpeaks) / sampling_rate * 1000.0
            hr = 60000.0 / rr_ms
            rr_diff = np.diff(rr_ms)

            out.update({
                "ecg_hr_mean": np.mean(hr),
                "ecg_hr_median": np.median(hr),
                "ecg_hr_std": np.std(hr),
                "ecg_hr_min": np.min(hr),
                "ecg_hr_max": np.max(hr),
                "ecg_hr_range": np.ptp(hr),
                "ecg_mean_nn": np.mean(rr_ms),
                "ecg_median_nn": np.median(rr_ms),
                "ecg_sdnn": np.std(rr_ms, ddof=1) if rr_ms.size > 1 else 0.0,
                "ecg_rmssd": np.sqrt(np.mean(rr_diff ** 2)) if rr_diff.size else 0.0,
                "ecg_sdsd": np.std(rr_diff, ddof=1) if rr_diff.size > 1 else 0.0,
                "ecg_pnn20": np.mean(np.abs(rr_diff) > 20) * 100 if rr_diff.size else 0.0,
                "ecg_pnn50": np.mean(np.abs(rr_diff) > 50) * 100 if rr_diff.size else 0.0,
                "ecg_rr_range": np.ptp(rr_ms),
                "ecg_rr_iqr": np.percentile(rr_ms, 75) - np.percentile(rr_ms, 25),
                "ecg_rr_cv": np.std(rr_ms) / (np.mean(rr_ms) + EPSILON),
                "ecg_rr_mad": stats.median_abs_deviation(rr_ms),
            })

            if rr_diff.size:
                sd1 = np.sqrt(0.5 * np.var(rr_diff))
                sd2 = np.sqrt(max(2 * np.var(rr_ms) - 0.5 * np.var(rr_diff), 0.0))
                out.update({
                    "ecg_sd1": sd1,
                    "ecg_sd2": sd2,
                    "ecg_sd1_sd2_ratio": sd1 / (sd2 + EPSILON),
                    "ecg_poincare_area": np.pi * sd1 * sd2,
                })

            out.update(nonlinear_features(rr_ms, prefix="ecg_rr_"))

            # HRV espectral sobre intervalos interpolados.
            if rr_ms.size >= 10:
                rr_time = np.cumsum(rr_ms) / 1000.0
                interp_fs = 4.0
                uniform_t = np.arange(rr_time[0], rr_time[-1], 1 / interp_fs)
                if uniform_t.size >= 16:
                    interp_rr = np.interp(uniform_t, rr_time, rr_ms)
                    hrv_spec = spectral_features(
                        interp_rr,
                        interp_fs,
                        prefix="ecg_hrv_",
                        bands={
                            "vlf": (0.0033, 0.04),
                            "lf": (0.04, 0.15),
                            "hf": (0.15, 0.40),
                        }
                    )
                    out.update(hrv_spec)
                    lf = out.get("ecg_hrv_power_lf", 0.0)
                    hf = out.get("ecg_hrv_power_hf", 0.0)
                    out["ecg_hrv_lf_hf_ratio"] = lf / (hf + EPSILON)

        return {k: safe_float(v) for k, v in out.items()}

    except Exception:
        return {k: safe_float(v) for k, v in out.items()}

In [20]:
def extract_eda_neurokit_only(eda: np.ndarray, sampling_rate: float = 700.0) -> Dict[str, float]:
    eda = sanitize_signal(eda)
    out = {}
    try:
        cleaned = nk.eda_clean(eda, sampling_rate=sampling_rate)
        decomposed = nk.eda_phasic(cleaned, sampling_rate=sampling_rate)

        tonic = sanitize_signal(decomposed["EDA_Tonic"].to_numpy())
        phasic = sanitize_signal(decomposed["EDA_Phasic"].to_numpy())

        out.update(general_features(tonic, sampling_rate, prefix="eda_tonic_"))
        out.update(general_features(phasic, sampling_rate, prefix="eda_phasic_"))
        out.update(nonlinear_features(tonic, prefix="eda_tonic_"))
        out.update(nonlinear_features(phasic, prefix="eda_phasic_"))
        out.update(wavelet_features(phasic, prefix="eda_phasic_"))

        _, info = nk.eda_peaks(phasic, sampling_rate=sampling_rate)

        peaks = np.asarray(info.get("SCR_Peaks", []), dtype=np.int64)
        onsets = np.asarray(info.get("SCR_Onsets", []), dtype=np.int64)
        amplitudes = np.asarray(info.get("SCR_Amplitude", []), dtype=np.float64)
        rise_times = np.asarray(info.get("SCR_RiseTime", []), dtype=np.float64)
        recovery_times = np.asarray(info.get("SCR_RecoveryTime", []), dtype=np.float64)

        amplitudes = amplitudes[np.isfinite(amplitudes)]
        rise_times = rise_times[np.isfinite(rise_times)]
        recovery_times = recovery_times[np.isfinite(recovery_times)]

        duration_min = eda.size / sampling_rate / 60.0

        out.update({
            "eda_scr_count": float(peaks.size),
            "eda_scr_rate_per_minute": peaks.size / (duration_min + EPSILON),
            "eda_scr_mean_amplitude": np.mean(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_median_amplitude": np.median(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_std_amplitude": np.std(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_max_amplitude": np.max(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_sum_amplitude": np.sum(amplitudes) if amplitudes.size else 0.0,
            "eda_scr_mean_rise_time": np.mean(rise_times) if rise_times.size else 0.0,
            "eda_scr_std_rise_time": np.std(rise_times) if rise_times.size else 0.0,
            "eda_scr_mean_recovery_time": np.mean(recovery_times) if recovery_times.size else 0.0,
            "eda_scr_std_recovery_time": np.std(recovery_times) if recovery_times.size else 0.0,
            "eda_phasic_auc": trapezoid(np.abs(phasic), dx=1 / sampling_rate),
            "eda_tonic_phasic_ratio": np.mean(np.abs(tonic)) / (np.mean(np.abs(phasic)) + EPSILON),
        })

        if peaks.size and onsets.size:
            n = min(peaks.size, onsets.size)
            widths = (peaks[:n] - onsets[:n]) / sampling_rate
            widths = widths[widths >= 0]
            out["eda_scr_mean_onset_to_peak"] = np.mean(widths) if widths.size else 0.0
            out["eda_scr_std_onset_to_peak"] = np.std(widths) if widths.size else 0.0

        return {k: safe_float(v) for k, v in out.items()}

    except Exception:
        return {k: safe_float(v) for k, v in out.items()}

## 6. Guardado seguro, lectura del checkpoint y comprobación de ventanas

In [21]:
def save_atomic(obj, path):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    with open(tmp,'wb') as f: pickle.dump(obj,f,protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp,path)
def read_pickle(path):
    with open(path,'rb') as f: return pickle.load(f)
def original_path(subject):return ORIGINAL_DIR/f'{subject}_features.pkl'
def extra_path(subject):return EXTRA_DIR/f'{subject}_neurokit_features.pkl'
def merged_path(subject):return MERGED_DIR/f'{subject}_features_full.pkl'
def assert_original(obj,subject):
    assert obj.get('subject')==subject,(subject,'identificador')
    X=obj['feature_matrix']['X']; names=list(obj['feature_matrix']['names']); meta=obj['metadata']; y=np.asarray(obj['target'])
    assert X.ndim==2 and X.shape[0]==len(y)==len(meta['starts'])==len(meta['ends'])
    assert X.shape[1]==len(names) and len(names)==len(set(names))
    starts=np.asarray(meta['starts']); ends=np.asarray(meta['ends'])
    assert np.all(ends>starts) and np.all(np.diff(starts)>=0)
    assert all(x in meta for x in ('original_label','purity'))
    return starts,ends,y

## 7. Procesar cada sujeto: índices exactos del checkpoint antiguo

Se leen únicamente las señales ECG y EDA del registro original, se recortan según `metadata.starts`/`metadata.ends` y se construye la matriz adicional. No se regeneran bloques ni etiquetas.

In [22]:
def calculate_extra(subject, max_windows=None):
    original=read_pickle(original_path(subject))
    starts,ends,y=assert_original(original,subject)
    fs=float(original['metadata'].get('sampling_rate',SAMPLING_RATE))
    raw=load_wesad_subject(BASE_PATH,subject)
    chest=raw['signal']['chest']; ecg=np.asarray(chest['ECG']).reshape(-1); eda=np.asarray(chest['EDA']).reshape(-1)
    assert int(ends.max())<=min(len(ecg),len(eda)), f'{subject}: índices fuera del registro crudo'
    limit=len(starts) if max_windows is None else min(len(starts),int(max_windows))
    if limit!=len(starts):print(f'ATENCIÓN: prueba parcial ({limit}/{len(starts)} ventanas); no fusionar como definitivo.')
    # Determinamos lista canónica sobre primera ventana. Para fallos posteriores se rellena NaN.
    rows=[]; errors=[]; name_order=[]; seen=set()
    for i in tqdm(range(limit),desc=f'{subject} NeuroKit2'):
        a,b=int(starts[i]),int(ends[i]); feat={}
        for modality,func,data in [('ECG',extract_ecg_neurokit_only,ecg),('EDA',extract_eda_neurokit_only,eda)]:
            try:
                result=func(data[a:b],sampling_rate=fs)
                if not result: errors.append((i,modality,'Sin características (fallo interno o señal insuficiente)'))
                feat.update(result)
            except Exception as exc:
                errors.append((i,modality,f'{type(exc).__name__}: {exc}'))
        rows.append(feat)
        for key in feat:
            if key not in seen:
                name_order.append(key);seen.add(key)
    # Orden de primera aparición; no suponemos que todas las ventanas tengan las mismas claves.
    assert name_order,f'{subject}: no se pudo extraer ninguna nueva columna. Verifica NeuroKit2 y señales.'
    matrix=np.full((limit,len(name_order)),np.nan,dtype=np.float32)
    for i,feat in enumerate(rows):
        for j,name in enumerate(name_order):
            if name in feat:
                val=feat[name]
                try: matrix[i,j]=float(val)
                except (ValueError,TypeError):pass
    # Reemplazar infinitos por NaN, pero NO imputar: eso pertenece al entrenamiento.
    matrix[~np.isfinite(matrix)]=np.nan
    result={'subject':subject,'feature_matrix':{'X':matrix,'names':name_order},
        'target':y[:limit].copy(),
        'metadata':{'starts':starts[:limit].copy(),'ends':ends[:limit].copy(),
                    'original_label':np.asarray(original['metadata']['original_label'])[:limit].copy(),
                    'purity':np.asarray(original['metadata']['purity'])[:limit].copy(),
                    'sampling_rate':fs,'n_segments':limit,
                    'source_checkpoint':str(original_path(subject)),
                    'extractor':'NeuroKit2 incremental — ramas originales ECG/EDA',
                    'partial':limit!=len(starts),'errors':errors[:100],
                    'total_errors':len(errors)}}
    del raw,chest,ecg,eda,rows,original
    gc.collect()
    return result

def run_extra(subjects=SUBJECTS_TO_PROCESS):
    for subject in subjects:
        dest=extra_path(subject)
        if dest.exists() and not OVERWRITE_EXTRA:
            print(subject,'YA EXISTE; se conserva:',dest)
            continue
        print('Extracción:',subject)
        result=calculate_extra(subject,TEST_MAX_WINDOWS)
        save_atomic(result,dest)
        X=result['feature_matrix']['X']
        print(subject,'guardado:',X.shape,'| NaN:',round(float(np.isnan(X).mean()),4),'| fallos:',result['metadata']['total_errors'])
        del result;gc.collect()

## 8. Ejecutar extracción incremental

**Consejo:** para validar el flujo, ejecuta primero `SUBJECTS_TO_PROCESS=["S2"]` y `TEST_MAX_WINDOWS=5` desde la configuración; luego borra ese archivo de prueba en `checkpoints_neurokit`, restaura la lista completa y `TEST_MAX_WINDOWS=None`. No elimines jamás los checkpoints originales.

In [ ]:
run_extra()

Extracción: S2


S2 NeuroKit2:   0%|          | 0/7767 [00:00<?, ?it/s]

S2 guardado: (7767, 187) | NaN: 0.0 | fallos: 0
Extracción: S3


S3 NeuroKit2:   0%|          | 0/7903 [00:00<?, ?it/s]

S3 guardado: (7903, 187) | NaN: 0.0002 | fallos: 0
Extracción: S4


S4 NeuroKit2:   0%|          | 0/7943 [00:00<?, ?it/s]

S4 guardado: (7943, 187) | NaN: 0.0002 | fallos: 0
Extracción: S5


S5 NeuroKit2:   0%|          | 0/8151 [00:00<?, ?it/s]

S5 guardado: (8151, 187) | NaN: 0.0 | fallos: 0
Extracción: S6


S6 NeuroKit2:   0%|          | 0/8091 [00:00<?, ?it/s]

## 9. Fusionar sujeto por sujeto — validación estricta de filas, nombres y etiquetas

In [ ]:
def merge_subject(subject):
    old=read_pickle(original_path(subject));new=read_pickle(extra_path(subject))
    starts,ends,y=assert_original(old,subject)
    X_old=old['feature_matrix']['X']; names_old=list(old['feature_matrix']['names'])
    X_new=np.asarray(new['feature_matrix']['X']);names_new=list(new['feature_matrix']['names'])
    assert not new['metadata'].get('partial'),f'{subject}: checkpoint adicional parcial, NO fusionar'
    assert X_new.shape[0]==X_old.shape[0],f'{subject}: ventanas desiguales'
    assert np.array_equal(starts,np.asarray(new['metadata']['starts'])),f'{subject}: starts no coinciden'
    assert np.array_equal(ends,np.asarray(new['metadata']['ends'])),f'{subject}: ends no coinciden'
    assert np.array_equal(y,np.asarray(new['target'])),f'{subject}: targets no coinciden'
    assert np.array_equal(np.asarray(old['metadata']['original_label']),np.asarray(new['metadata']['original_label']))
    assert np.allclose(np.asarray(old['metadata']['purity']),np.asarray(new['metadata']['purity']))
    overlap=set(names_old)&set(names_new)
    assert not overlap,f'{subject}: nombres duplicados: {sorted(overlap)[:10]}'
    assert len(names_new)==len(set(names_new))
    merged=dict(old)
    merged['feature_matrix']={'X':np.concatenate([X_old,X_new],axis=1),'names':names_old+names_new}
    merged['metadata']=dict(old['metadata'])
    merged['metadata']['n_features']=merged['feature_matrix']['X'].shape[1]
    merged['metadata']['neurokit_incremental']={
        'n_added':len(names_new),'source_extra':str(extra_path(subject)),
        'extra_nan_cells':int(np.isnan(X_new).sum()),'extra_total_cells':int(X_new.size),
        'original_columns':len(names_old)}
    save_atomic(merged,merged_path(subject))
    return {'subject':subject,'windows':len(y),'original':len(names_old),'added':len(names_new),
            'total':len(names_old)+len(names_new),'extra_missing_%':round(100*float(np.isnan(X_new).mean()),2)}

def merge_all(subjects=SUBJECTS_TO_PROCESS):
    report=[]
    for s in subjects:
        if not extra_path(s).exists():
            print('FALTA extra:',s);continue
        try:
            report.append(merge_subject(s))
        except Exception as exc:
            print('NO FUSIONADO:',s,type(exc).__name__,str(exc))
    df=pd.DataFrame(report)
    display(df)
    df.to_csv(OUTPUT_DIR/'reporte_fusion_neurokit.csv',index=False)
    return df

## 10. Ejecutar fusión y comprobaciones

In [ ]:
fusion_report=merge_all()

## 11. Consolidar checkpoints finales sin alterar los originales

In [ ]:
def consolidate_full(subjects=SUBJECTS):
    missing=[s for s in subjects if not merged_path(s).exists()]
    if missing:raise FileNotFoundError(f'No se consolida: faltan archivos fusionados de {missing}')
    obj={'_global_metadata':{'subjects_requested':list(subjects),'subjects_available':list(subjects),
                              'processing':'Fusion de checkpoints originales + nuevas columnas NeuroKit2'}}
    # Unificar esquema de columnas entre sujetos sin cambiar el orden de las filas.
    union=[];seen=set()
    for s in subjects:
        item=read_pickle(merged_path(s))
        for name in item['feature_matrix']['names']:
            if name not in seen:union.append(name);seen.add(name)
        del item
    print('Esquema común:',len(union),'columnas')
    col_index={name:i for i,name in enumerate(union)}
    for s in subjects:
        item=read_pickle(merged_path(s))
        X=np.asarray(item['feature_matrix']['X'])
        old_names=item['feature_matrix']['names']
        if old_names!=union:
            aligned=np.full((X.shape[0],len(union)),np.nan,dtype=np.float32)
            for j,name in enumerate(old_names):aligned[:,col_index[name]]=X[:,j]
            item['feature_matrix']={'X':aligned,'names':list(union)}
        obj[s]=item
    save_atomic(obj,FINAL_OUTPUT)
    print('CONSOLIDADO GUARDADO:',FINAL_OUTPUT)
    print('Tamaño:',round(FINAL_OUTPUT.stat().st_size/1024**2,1),'MiB')
    return FINAL_OUTPUT

# No ejecutar la consolidación en modo de prueba parcial.
if TEST_MAX_WINDOWS is None:
    final_file=consolidate_full()
else:
    print('Modo prueba: consolidación final omitida.')

## 12. Catálogo efectivo: nombres exactos de las columnas, por modalidad

In [ ]:
if merged_path('S2').exists():
    subj=read_pickle(merged_path('S2'))
    names=subj['feature_matrix']['names']
    rows=[]
    for prefix in ['ecg_','eda_','resp_']:
        signal_name=prefix[:-1].upper()
        sel=[x for x in names if x.startswith(prefix)]
        for i,name in enumerate(sel,1):
            rows.append({'codigo':f'{signal_name}_{i}','senal':signal_name,'feature_name':name,
                         'origen':'adicional NeuroKit2' if name in read_pickle(extra_path('S2'))['feature_matrix']['names'] else 'checkpoint original'})
    catalog=pd.DataFrame(rows)
    catalog.to_csv(OUTPUT_DIR/'feature_catalog_ecg_eda_resp_con_neurokit.csv',index=False,encoding='utf-8-sig')
    print(catalog.groupby(['senal','origen']).size().to_string())
    print('Catálogo:',OUTPUT_DIR/'feature_catalog_ecg_eda_resp_con_neurokit.csv')
else:
    print('No está disponible el checkpoint fusionado de S2.')

## Observaciones para el capítulo de tesis

1. El número real de nuevas columnas se determina **después de ejecutar el notebook**. Los cálculos de picos y separación EDA pueden fallar en algunas ventanas. Las columnas con NaN no deben imputarse utilizando los sujetos de prueba.
2. Algunas funciones de la libreta original capturan excepciones y devuelven un diccionario parcial. Por eso el reporte de NaN es indispensable y la ausencia de una columna no debe confundirse con un valor fisiológico cero.
3. La ejecución debe conservar las ventanas de **60 s, avance de 250 ms y etiquetas [1,3] vs [2]**, porque se usan los índices originales; no se altera el protocolo.
4. La lista final de nombres se exporta a CSV para actualizar el diccionario de la tesis **a partir de la salida real**, no de un inventario teórico.
5. Los archivos pickle se deben abrir solo si proceden de un origen confiable: Python puede ejecutar código al deserializarlos.